[**Apache Airflow**](https://airflow.apache.org/) is a workflow orchestration platform used to programmatically author, schedule, and monitor pipelines.

### Initial Setup

In [2]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import time

In [3]:
# Tells Airflow where to store airflow.db, logs, airflow.cfg
# ONE AIRFLOW home per project
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="/home/zephyr/workspace/ml_toolbox/MLOps/Airflow/.env")

print("AIRFLOW_HOME:", os.environ["AIRFLOW_HOME"])
# Explicitly disable examples
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"

AIRFLOW_HOME: /home/zephyr/workspace/ml_toolbox/MLOps/Airflow


In [ ]:
# !export $(grep -v '^#' .env | xargs) # load env vars

In [ ]:
# !airflow db init # create sqlite db file

/home/zephyr/workspace/ml_toolbox/MLOps/Airflow/.venv_airflow/lib/python3.12/site-packages/airflow/utils/providers_configuration_loader.py:55 DeprecationWarning: `db init` is deprecated.  Use `db migrate` instead to migrate the db and/or airflow connections create-default-connections to create the default connections
DB: sqlite:////home/zephyr/workspace/ml_toolbox/MLOps/Airflow/airflow.db
[2026-01-26T14:36:11.852+0400] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-01-26T14:36:11.853+0400] {migration.py:210} INFO - Will assume non-transactional DDL.
[2026-01-26T14:36:11.927+0400] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-01-26T14:36:11.927+0400] {migration.py:210} INFO - Will assume non-transactional DDL.
[2026-01-26T14:36:11.928+0400] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-01-26T14:36:11.928+0400] {migration.py:210} INFO - Will assume non-transactional DDL.
[2026-01-26T14:36:11.929+0400] {db.py:1675} INFO - Creating tables
INFO  [alembic.runt

In [ ]:
# airflow webserver --port ****
# airflow scheduler  # start scheduler
# airflow users list # check user list
# airflow users delete --username *****
# airflow db migrate # reconnect db

`create user account`
```bash
airflow users create \
    --username admin \
    --firstname Admin \
    --lastname User \
    --role Admin \
    --email admin@example.com \
    --password YourNewPassword123
```

### What is a DAG?
A **DAG (Directed Acyclic Graph)** is a collection of all the tasks you want to run, organized in a way that reflects their relationships and dependencies.

- **Directed** → tasks have a direction (A → B)
- **Acyclic** → no loops (a task cannot depend on itself, directly or indirectly)
- **Graph** → tasks are nodes, dependencies are edges

A DAG does **not** execute code itself — it only **describes** the workflow.  
Airflow’s scheduler reads the DAG and decides *when* and *how* tasks should run.

In [4]:
# Example DAG workflow

def extract():
    print("Extracting data")
def transform():
    print("Transforming data")
def load():
    print("Loading data")

# define args for DAG
default_args = {
    "owner": "data-team",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
}

# create DAG
with DAG(
    dag_id="example_etl_dag",
    description="A simple ETL DAG for learning Airflow",
    start_date=datetime(2024, 1, 1),
    schedule="@daily",
    catchup=False,
    default_args=default_args,
) as dag:
    # python tasks
    extract_task = PythonOperator(
        task_id="extract_data",
        python_callable=extract,
    )
    transform_task = PythonOperator(
        task_id="transform_data",
        python_callable=transform,
    )
    load_task = PythonOperator(
        task_id="load_data",
        python_callable=load,
    )

    # define the task dependency
    extract_task >> transform_task >> load_task

### Airflow Project Components

```text
airflow-project/
│
├── dags/
│   ├── example_etl_dag.py
│   ├── ml_training_dag.py
│   └── ...
│
├── logs/
│   ├── example_etl_dag/
│   │   └── extract_data/
│   │       └── 2024-01-02/
│   │           └── 1.log
│
├── plugins/
│   └── custom_operators.py
│
├── airflow.cfg
├── airflow.db
├── airflow-webserver.pid
└── webserver_config.py
```

1. **DAG files (`dags/`)**
- contains workflow definitions which are parsed by scheduler, no execution takes place here

In [5]:
!ls -laR dags

dags:
total 20
drwxr-xr-x 3 zephyr zephyr 4096 Jan 26 14:35 .
drwxr-xr-x 5 zephyr zephyr 4096 Jan 26 15:16 ..
-rw-r--r-- 1 zephyr zephyr  760 Jan 25 23:50 01_basic_bash.py
-rw-r--r-- 1 zephyr zephyr  520 Jan 25 23:57 02_python_operator.py
drwxr-xr-x 2 zephyr zephyr 4096 Jan 26 14:35 __pycache__

dags/__pycache__:
total 12
drwxr-xr-x 2 zephyr zephyr 4096 Jan 26 14:35 .
drwxr-xr-x 3 zephyr zephyr 4096 Jan 26 14:35 ..
-rw-r--r-- 1 zephyr zephyr  979 Jan 26 14:35 01_basic_bash.cpython-312.pyc


2. **Task execution & logs (`logs/`)**
- Logs are written per task instance
- Path structure mirrors: DAG → task_id → execution_date → try_number

In [6]:
!ls -laR logs

logs:
total 16
drwxr-xr-x 4 zephyr zephyr 4096 Jan 26 14:35 .
drwxr-xr-x 5 zephyr zephyr 4096 Jan 26 15:16 ..
drwxr-xr-x 2 zephyr zephyr 4096 Jan 26 14:35 dag_processor_manager
drwxr-xr-x 4 zephyr zephyr 4096 Jan 26 14:30 scheduler

logs/dag_processor_manager:
total 28
drwxr-xr-x 2 zephyr zephyr  4096 Jan 26 14:35 .
drwxr-xr-x 4 zephyr zephyr  4096 Jan 26 14:35 ..
-rw-r--r-- 1 zephyr zephyr 17287 Jan 26 14:35 dag_processor_manager.log

logs/scheduler:
total 16
drwxr-xr-x 4 zephyr zephyr 4096 Jan 26 14:30 .
drwxr-xr-x 4 zephyr zephyr 4096 Jan 26 14:35 ..
drwxr-xr-x 2 zephyr zephyr 4096 Jan 25 23:17 2026-01-25
drwxr-xr-x 3 zephyr zephyr 4096 Jan 26 14:35 2026-01-26
lrwxrwxrwx 1 zephyr zephyr   10 Jan 26 14:30 latest -> 2026-01-26

logs/scheduler/2026-01-25:
total 8
drwxr-xr-x 2 zephyr zephyr 4096 Jan 25 23:17 .
drwxr-xr-x 4 zephyr zephyr 4096 Jan 26 14:30 ..

logs/scheduler/2026-01-26:
total 16
drwxr-xr-x 3 zephyr zephyr 4096 Jan 26 14:35 .
drwxr-xr-x 4 zephyr zephyr 4096 Jan 26 14:30 ..

3. **`airflow.db`**
It is a SQLite metadata database (default for local setups) for storing
- DAGs
- DAG runs
- Task instances
- States (success, failed, running)

It is not for production (we use other dbs like Postgres/MySQL instead)

If this file is deleted 
- All DAG history is lost
- DAG definitions will be reloaded from code

4. `airflow.cfg` file

This is the main Airflow configuration file. It Controls:
- Executor type
- DAG folder location
- Logging behavior
- Webserver and scheduler settings

5. `airflow-webserver.pid`
It stores the process ID of the running webserver and used to:
- Stop the webserver cleanly
- Detect if it’s already running

If Airflow thinks the webserver is running when it’s not, deleting this file usually fixes it

6. `webserver_config.py`
This is advanced configuration for the Airflow UI:

It Controls:
- Authentication
- RBAC
- UI behavior
Usually there is no requirement to modify this.

### How Airflow Finds DAGs

Airflow does not “register” DAGs manually.

Instead:

- we write `.py` files
- The **scheduler scans the DAGs folder**
- Every few seconds, Airflow:
  - imports each Python file
  - looks for `DAG` objects
  - stores metadata in the database


In [7]:
# In Airflow, we write .py files that the scheduler scans.
# Below is our DAGs folder

DAGS_FOLDER = os.path.join(os.getcwd(), "dags")
os.makedirs(DAGS_FOLDER, exist_ok=True)
DAGS_FOLDER

'/home/zephyr/workspace/ml_toolbox/MLOps/Airflow/dags'

> If a DAG file is **not inside the DAGs folder**, Airflow will ignore it.

Therefore, 
- File paths matter
- Syntax errors can silently break DAG discovery

### DAG ID vs Task ID


`dag_id`
- A unique identifier for the entire workflow
- Defined at the DAG level
- Must be globally unique across the Airflow environment
- Used in:
  - Airflow UI
  - CLI commands
  - Logs and metadata database

Example:
```python
dag_id="example_etl_pipeline"
```

---

`task_id`
- A unique identifier for a task within a DAG
- Must be unique inside its DAG
- Can be reused across different DAGs
Example:
```python
task_id="extract_data"
```


### Scheduling

``start_date``
```python
start_date=datetime(2024, 1, 1)
```
- Defines when Airflow starts scheduling DAG runs
- Airflow schedules based on logical dates, not current time
- No tasks will run for dates before this

``end_date``
- when to stop running the DAG instance

``max_tries``
- how many retry attempts to make if run failed

``schedule``
```python
schedule_interval="@daily"
```
- Tells Airflow to create one DAG run per day
- Can use presets (`@daily`, `@hourly`) or `cron expressions`
- Use `None` for manual-only DAGs

`catchup`
```python
catchup=False
```
- Prevents Airflow from backfilling all missed runs
- Recommended for most production DAGs

**Ways to define Task Dependencies**
```
t1 >> t2
[t1, t2] >> t3
t1.set_downstream(t2)
```

### Others Params

```python
default_args = {
    "owner": "data-team",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
}
```
- we can use it to share default settings for all tasks
- Reduces repetition
- Task-level arguments override these values

### Airflow CLI & Terminal Commands

```bash

### List all DAGs
airflow dags list
```

It shows all DAGs Airflow has discovered. 

DAGs appear here only if:
- The .py file is in the dags/ folder
- The file has no syntax errors
- A DAG object is created successfully

If your DAG doesn’t show up here, Airflow hasn’t loaded it.

In [8]:
!airflow dags list

/home/zephyr/workspace/ml_toolbox/MLOps/Airflow/.venv_airflow/lib/python3.12/site-packages/airflow/cli/commands/dag_command.py:48 UserWarning: Could not import graphviz. Rendering graph to the graphical format will not be possible.
dag_id   | fileloc                                          | owners | is_paused
=========+==================================================+========+==========
basic_v1 | /home/zephyr/workspace/ml_toolbox/MLOps/Airflow/ | zephyr | True     
         | dags/01_basic_bash.py                            |        |          
                                                                                


```bash
### List DAG runs
airflow dags list-runs -d dag_id
```
- Displays all executions (DAG runs) for a DAG
- Each row corresponds to one logical date
- Useful for debugging scheduling and catchup behavior

In [11]:
!airflow dags list-runs -d basic_v1

/home/zephyr/workspace/ml_toolbox/MLOps/Airflow/.venv_airflow/lib/python3.12/site-packages/airflow/cli/commands/dag_command.py:48 UserWarning: Could not import graphviz. Rendering graph to the graphical format will not be possible.
No data found


```bash
### List tasks in a DAG
airflow tasks list dag_id
```
- Shows all task_ids defined in the DAG
- Confirms Airflow correctly parsed your task definitions

In [13]:
!airflow tasks list basic_v1

create_temp_dir
print_hello



```bash
### Debug
airflow tasks test example_etl_dag extract_data 2024-01-02
```
- Runs one task only
- Ignores dependencies
- Does NOT create a DAG run in the UI
- Runs the task code locally and prints logs to the terminal
- is one of the most useful debugging commands in Airflow.

In [17]:
# !airflow tasks test basic_v1 print_hello 2024-01-02
# !airflow tasks test basic_v1 create_temp_dir 2024-01-02

In [ ]:
# In Airflow, we write .py files that the scheduler 'scans'.
# below is our dags folder
DAGS_FOLDER = os.path.join(os.getcwd(), "dags")
os.makedirs(DAGS_FOLDER, exist_ok=True)
DAGS_FOLDER

### Tasks & Operators

- **Operator** → defines *what type of work* is done  
- **Task** → a specific instance of an operator inside a DAG  

Each task:
- has a `task_id`
- uses exactly **one operator**
- represents a **single unit of work**

---

#### Python Operator
- Execute python functions 


In [18]:
from airflow.operators.python import PythonOperator

def transform_data(table_name, multiplier, **kwargs):
    print(f"Transforming table: {table_name}")
    print(f"Multiplier: {multiplier}")

    # kwargs can contain Airflow context
    print("Extra kwargs received:")
    for key, value in kwargs.items():
        print(f"{key} = {value}")


transform_task = PythonOperator(
    task_id="transform_data",
    python_callable=transform_data,
    op_args=["sales_table", 10],              # positional arguments
    op_kwargs={"env": "dev", "owner": "data"},# keyword arguments
)

#### Bash Operator
- runs shell commands directly

Example use cases:
- Running scripts (.sh, .py)
- dbt commands (dbt run, dbt test)
- File operations
- System utilities

In [ ]:
from airflow.operators.bash import BashOperator

bash_task = BashOperator(
    task_id="run_shell_command",
    bash_command="echo 'Hello from Airflow'",
)

#### Email Operator
- send emails based on task outcomes

In [ ]:
from airflow.operators.email import EmailOperator

email_task = EmailOperator(
    task_id="send_notification",
    to="data-team@example.com",
    subject="Airflow DAG Completed",
    html_content="The DAG has finished successfully.",
)

#### Airflow Sensors

they are operators that waits for an event to be true in order to run

They listen for events such as:
- file creation
- another DAG to finish
- upload a db record
- response to web request

``mode``
- ``poke`` mode (default): this occupies a worker slot, therfore, not very efficient
- ``reschedule`` mode: checks for event, if not activate, it gives up the queue and re-queue for next try (recommended)

``poke_interval``
- how often the sensor checks for condition
```python
poke_interval=30
```

``timeout``
- max time to wait before failing
```python
timeout=60*10 # 10 minutes
```


**Debugging Sensors**

```bash
# Logs will show each poke attempt.
airflow tasks test file_sensor_example wait_for_input_file 2024-01-02
```

In [ ]:
# File Sensor
from airflow.sensors.filesystem import FileSensor

# DAG starts and check if input.csv file exists
# if exists, run the process_file task, otherwise, wait 30 seconds and retry
with DAG(
    dag_id="file_sensor_example",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:

    wait_for_file = FileSensor(
        task_id="wait_for_input_file",
        filepath="data/input.csv",
        fs_conn_id="fs_default",
        poke_interval=30,
        timeout=300,
        mode="reschedule",
    )

    process_file = PythonOperator(
        task_id="process_file",
        python_callable=lambda: print("File detected, processing started"),
    )

    wait_for_file >> process_file

In [23]:
from airflow.sensors.date_time import DateTimeSensor

with DAG(
    dag_id="datetime_sensor_test",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:

    wait_until_time = DateTimeSensor(
        task_id="wait_until_9am",
        target_time="2024-01-02T09:00:00",
        poke_interval=60,
        mode="reschedule",
    )

In [28]:
!airflow tasks test datetime_sensor_test wait_until_9am 2024-01-02

[2026-01-26T23:21:26.159+0400] {dagbag.py:588} INFO - Filling up the DagBag from /home/zephyr/workspace/ml_toolbox/MLOps/Airflow/dags
[2026-01-26T23:21:26.308+0400] {taskinstance.py:2613} INFO - Dependencies all met for dep_context=non-requeueable deps ti=<TaskInstance: datetime_sensor_test.wait_until_9am __airflow_temporary_run_2026-01-26T19:21:26.232637+00:00__ [None]>
[2026-01-26T23:21:26.313+0400] {taskinstance.py:2613} INFO - Dependencies all met for dep_context=requeueable deps ti=<TaskInstance: datetime_sensor_test.wait_until_9am __airflow_temporary_run_2026-01-26T19:21:26.232637+00:00__ [None]>
[2026-01-26T23:21:26.313+0400] {taskinstance.py:2866} INFO - Starting attempt 0 of 1
[2026-01-26T23:21:26.314+0400] {taskinstance.py:2947} WARNING - cannot record queued_duration for task wait_until_9am because previous state change time has not been saved
[2026-01-26T23:21:26.314+0400] {taskinstance.py:2889} INFO - Executing <Task(DateTimeSensor): wait_until_9am> on 2024-01-02 00:00:00+

Sensors do not remember past runs. Therefore, if we run this after "2024-01-02T09:00:00", it will be successful immediately since the target time has already passed.

In [ ]:
# External Task Sensor
from airflow.sensors.external_task import ExternalTaskSensor

wait_for_upstream_dag = ExternalTaskSensor(
    task_id="wait_for_other_dag",
    external_dag_id="upstream_dag",
    external_task_id=None,  # wait for entire DAG
    poke_interval=60,
    timeout=3600,
    mode="reschedule",
)

### Executors

An Executor defines how and where tasks are run. The Scheduler decides what should run and when. The Executor decides how tasks are executed.

DAG → Scheduler → Executor → Task Execution

Changing the executor does not change your DAG code — it changes how tasks are dispatched behind the scenes.

Executors determine:

- Parallelism
- Scalability
- Fault tolerance
- Resource isolation
- Production readiness

Same DAG, different executor = completely different runtime behavior.

**Common Airflow Executors**

- `SequentialExecutor`: 
    - no parallelism, even when tasks have no dependency they run one by one,
    - for local env or debugging only, uses SQLite

- `LocalExecutor`: 
    - achieves concurrency through parallelism by spawning separate processes, not multi-threading
    - Requires non-SQLite DB (Postgres / MySQL)
    - useful for medium workloads on single node production 
        ```text
        Task A ─┐
        Task B ─┼──> Run in parallel
        Task C ─┘
        ```

- `CeleryExecutor`:
    - distributed execution, medium-large scale production
    - Uses message broker (Redis / RabbitMQ)
    - workers run on multiple machines
        ```text
            Webserver → Scheduler → Broker → Workers (1,2,3...) → Tasks
        ```

- `KubernetesExecutor`:
    - fully dymanic, cloud-native, Elastic scaling
    - Each task runs in its own Kubernetes pod
    - useful for MLOps pipelines, data science workloads

**How to Switch Executors**

1. In `airflow.cfg` file >>> executor = SequentialExecutor

2. via environment variable >>> export AIRFLOW_CORE_EXECUTOR=SequentialExecutor


### Callbacks
callbacks are supported with all executors (Local, Celery, Kubernetes) but where and how they run depends on the executor.

**Common Callbacks**
- `on_success_callback`
- `on_failure_callback`
- `on_retry_callback`
- `on_execute_callback`


In [4]:
# on success callback
def my_func():
    pass
def notify_success(context):
    print(f"Task {context['task_instance'].task_id} succeeded")

task = PythonOperator(
    task_id="example",
    python_callable=my_func,
    on_success_callback=notify_success,
)

Parallel executors do not share memory or state. Therefore, callbacks:

- Cannot rely on global variables
- Cannot assume shared filesystem
- Cannot assume local services
- Cannot assume same machine

They must be: Stateless, Idempotent and Network-safe

```text
Scheduler
   |
Executor sends task to worker
   |
Task runs
   |
Callback executes on SAME worker/pod
```

Since they ran on same worker/pod, in distributed systems, using callback for local filesystem writes, in-memory caches and global variables may break as pods or environment is ephemeral and has no knowledge of other tasks.

### SLAs & Reporting in Airflow

An SLA defines how long a task is allowed to take after its scheduled start time. If the task does not finish within that duration:
- The task is marked as SLA missed
- An SLA miss event is recorded
- Optional notifications (email / callback) are triggered

Example
```text
DAG scheduled at: 09:00
sla = 30 minutes
deadline = 09:30
task finishes at 09:35 → SLA MISS
```

In [6]:
# define task level SLA

# tasks sleeps for 40 seconds 
def slow_task():
    time.sleep(40)

with DAG(
    dag_id="sla_task_example",
    start_date=datetime(2024, 1, 1),
    schedule="@daily",
    catchup=False,
) as dag:
    # our SLA is only 30 seconds
    task_with_sla = PythonOperator(
        task_id="slow_task",
        python_callable=slow_task,
        sla=timedelta(seconds=30),
    )

# therefore, SLA miss is recorded
# we can check it in web UI

Task-level SLA overrides DAG-level SLA if both exist.

In [ ]:
# DAG level SLA

default_args = {
    "sla": timedelta(minutes=10),
}
with DAG(
    dag_id="sla_dag_level",
    default_args=default_args,
    start_date=datetime(2024, 1, 1),
    schedule="@daily",
    catchup=False,
):
    pass


SLA misses are checked by the Scheduler periodically (not real-time). Therefore, SLA alerts may arrive a few minutes late.

In [ ]:
def sla_miss_callback(dag, task_list, blocking_task_list, slas, blocking_tis):
    '''
        dag:	    DAG object
        task_list:  Tasks that missed SLA
        blocking_task_list:     Tasks blocking others
        slas:           SLA definitions
        blocking_tis:   TaskInstances causing delays
    '''
    print("SLA MISSED!")
    print(f"DAG: {dag.dag_id}")
    print(f"Tasks missed SLA: {task_list}")

with DAG(
    dag_id="sla_callback_example",
    start_date=datetime(2024, 1, 1),
    schedule="@daily",
    catchup=False,
    sla_miss_callback=sla_miss_callback,
):
    ...

In [12]:
# Example: SLA + Email + Retry + Failure

def flaky_task():
    time.sleep(35)
    raise ValueError("Task failed")

def notify_sla_miss(dag, task_list, blocking_task_list, slas, blocking_tis):
    print("SLA MISS ALERT")
    print(task_list)

default_args = {
    "owner": "data-team",
    "retries": 2,
    "retry_delay": timedelta(minutes=1),
    "email": ["data-alerts@example.com"],
    "email_on_failure": True,
    "email_on_retry": True,
}

with DAG(
    dag_id="sla_full_example",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    default_args=default_args,
    sla_miss_callback=notify_sla_miss,
) as dag:

    task = PythonOperator(
        task_id="flaky_task",
        python_callable=flaky_task,
        sla=timedelta(seconds=30),
    )